# COMP9085 Week 7 Laboratory
## PCA and the geometry of a learned representation

**Two hours.** Until now you have chosen the features. This week you meet two ways of
*replacing* them: principal component analysis (PCA), which finds a few directions that
summarise many measured features, and a protein **foundation model**, which turns a raw
sequence into a vector it learned to produce from millions of unlabelled proteins. You will
reduce a real dataset properly, look inside a real learned representation, and then test,
with one fixed classifier, what that representation sees that simple features do not.

**Part A (Tasks 1–5, about 50 minutes): WDBC.** The breast-mass data from Weeks 2–4
(569 fine-needle aspirates, 30 nuclear measurements), loaded from scikit-learn's bundled copy.

**Part B (Tasks 6–10, about 60 minutes): human proteins.** 2,000 reviewed human proteins
from UniProtKB/Swiss-Prot (release 2026_03), one per UniRef50 cluster, each
with two real annotations (is it an integral **membrane** protein? does it carry an
N-terminal **signal peptide**?) and a real 320-dimensional embedding from the protein
language model **ESM-2** (8 million parameters). The module team computed the embeddings
once; you load them from `data/`. **No network, no GPU and no deep-learning library are
needed.** `data/README.md` documents every column and `data/PROVENANCE.json` the sources,
model version and checksums.

**The question.** When does a representation learned by a foundation model beat features
a biologist would choose, and *why*? A higher score is not an answer. An answer says what
property of the task the representation captures.

**By the end of this lab you should be able to**
1. explain the curse of dimensionality and show distance concentration numerically;
2. perform PCA correctly (standardised, inside a pipeline when it feeds a model) and
   interpret the explained-variance spectrum and the loadings;
3. distinguish what PCA preserves from what a neighbour embedding (t-SNE) preserves, and
   read each plot without over-reading it;
4. describe how a protein language-model embedding is produced, and measure two properties
   of its geometry (anisotropy, effective dimension);
5. compare engineered and learned representations under one fixed classifier and one
   protocol, and use a control experiment to explain *why* they differ.

**Prerequisites.** Weeks 3–6: train/test splits, `StratifiedKFold`, `Pipeline`, baselines,
ROC AUC and average precision. No biology beyond the two short explanations in Part B.

**How the cells work.** Cells marked *provided* run as they are: read them, run them, and
make sure you understand them. Cells containing `# your code here` are yours. Before some
tasks you are asked to **predict** a result first: write the prediction down before you
run anything.

**Structure.** **Tasks 1–10 are the core lab.** A checkpoint after Task 10 marks the end of
the core; three optional extensions and a final reflection follow.

**Rules for the whole lab.** Each dataset is split once, in a provided cell. Every choice is
made with 5-fold cross-validation on the training rows. Each test set is used **once**
(Task 5 and Task 10). Anything that learns from data (scaler, PCA, classifier) is fitted
inside a `Pipeline`, so cross-validation refits it in every fold.

**Practice, not assessment.** This lab is not part of Assignment 1 and uses different
data. Do not add PCA or embeddings to your assignment because they appear here; use a
method there only if you can justify it for *your* data, with *your* numbers.

**The Week 7 page is your interactive companion.** Its PCA visualiser lets you rotate and
project WDBC yourself; use it alongside Tasks 2–4.

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**), with
every question answered in the markdown cells using your own numbers.

### Setup

Run this cell first.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import sklearn
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
DATA = Path("data")


def cv_scores(model, X, y, metrics=("roc_auc", "average_precision")):
    """Mean 5-fold CV score of a pipeline on the TRAINING rows (the course protocol)."""
    s = cross_validate(model, X, y, cv=CV, scoring=list(metrics))
    out = {m: s[f"test_{m}"].mean() for m in metrics}
    out["roc_auc_sd"] = s["test_roc_auc"].std(ddof=1) if "roc_auc" in metrics else np.nan
    return out

# Part A — Principal component analysis on WDBC

---
## Task 1 — The curse of dimensionality (provided, about 8 minutes)

Many biological datasets are *wide*: a transcriptome measures 20,000 genes in perhaps 200
patients. The cell below draws 1,000 points uniformly at random in a cube of dimension
*d* and measures the distance from the first point to every other point.

**Predict first.** As *d* grows from 2 to 1,000, what happens to the ratio between the
farthest and the nearest of those distances? Write your prediction here, then run the cell.

*(Prediction.)*

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for d in [2, 10, 100, 1000]:
    P = rng.random((1000, d))
    dist = np.linalg.norm(P[1:] - P[0], axis=1)
    rows.append({"dimension": d, "nearest": dist.min(), "farthest": dist.max(),
                 "relative contrast (far - near) / near": (dist.max() - dist.min()) / dist.min()})
pd.DataFrame(rows).set_index("dimension").round(3)

**Question 1.1.** Describe what happens to the nearest and farthest distances, and to
their relative contrast, as the dimension grows. Why does this matter for any method that
relies on "the nearest points" (next week: k-nearest neighbours)?

**Question 1.2.** WDBC has 30 features and 569 cases; a transcriptome study might have 20,000
features and 200 cases. Give one reason why reducing the dimension can help, and one thing
you could lose by doing it.

*Your answer here.*

---
## Task 2 — PCA on standardised WDBC (about 10 minutes)

The provided cell loads WDBC with **malignant = 1**, as in Weeks 2–4, and makes the one
split used in Part A (stratified, 30% test, `SEED = 42`, the Week 4 protocol).

In this task you are **describing** the data, not evaluating a model, so PCA is fitted on
all 569 cases and no label is used. When PCA feeds a model (Task 4) it goes inside the
pipeline instead.

In [ ]:
wdbc = load_breast_cancer()
X = wdbc.data
y = (wdbc.target == 0).astype(int)          # 1 = malignant
feature_names = list(wdbc.feature_names)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
print(X.shape, "| malignant:", y.sum(), "| train", len(y_train), "| test", len(y_test))

**Your task.** Standardise `X` with `StandardScaler`, fit `PCA()` (all components), and
plot (a) the explained-variance ratio of each component (a *scree plot*) and (b) the
cumulative explained variance, with a horizontal line at 0.90. Store the fitted PCA as
`pca_all` and print how many components are needed to reach 90% and 95%.

*Hint:* `pca_all.explained_variance_ratio_` and `np.cumsum`.

In [ ]:
# your code here


**Question 2.1.** How much variance do the first two components carry, and how many
components do you need for 90%? What does that say about how much a two-dimensional plot
of WDBC can show?

**Question 2.2.** WDBC measures 30 features, but ten of them are the *mean*, ten the
*standard error* and ten the *worst* value of the same ten nuclear properties. Why would you
expect such data to be compressible?

*Your answer here.*

---
## Task 3 — What the components mean, and why scaling matters (about 12 minutes)

A component is a weighted sum of the original features. The weights are its **loadings**.
The provided cell shows the six largest loadings (by absolute value) of PC1 and PC2. The sign
of a component is arbitrary: PCA could return PC1 or −PC1, so read the *pattern* of signs,
not the signs themselves.

In [ ]:
Z = StandardScaler().fit_transform(X)
pca2 = PCA().fit(Z)
for i in [0, 1]:
    load = pd.Series(pca2.components_[i], index=feature_names)
    top = load.reindex(load.abs().sort_values(ascending=False).index)[:6]
    print(f"PC{i + 1} ({pca2.explained_variance_ratio_[i]:.1%} of variance)  "
          f"| loadings with the same sign: {max((load > 0).sum(), (load < 0).sum())} of 30")
    print(top.round(3).to_string(), "\n")

scores = pca2.transform(Z)[:, :2]
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for lab, col, name in [(0, "#4a90c2", "benign"), (1, "#c8102e", "malignant")]:
    ax.scatter(scores[y == lab, 0], scores[y == lab, 1], s=8, alpha=0.6, c=col, label=name)
ax.set(xlabel="PC1", ylabel="PC2", title="WDBC projected on PC1 and PC2"); ax.legend()
plt.show()

**Predict first.** If you run PCA on the **unscaled** features, what share of the
variance do you expect PC1 to take, and which feature will dominate it? (Look at the units:
`worst area` is in square pixels, `mean smoothness` is a small dimensionless number.)

**Your task.** Fit `PCA()` on the unscaled `X`. Print PC1's share of the variance and the
feature with the largest absolute loading on PC1. Then print the standard deviations of
`worst area` and `mean smoothness`.

*(Prediction.)*

In [ ]:
# your code here


**Question 3.1.** All 30 loadings of PC1 have the same sign. Describe in words what a
case with a high score on PC1 looks like (use the features with the largest loadings), and
relate it to what a pathologist looks for in a malignant nucleus.

**Question 3.2.** PC2 contrasts the fractal-dimension features with the size features. What
kind of variation does it describe, and does it separate the two diagnoses in the plot?

**Question 3.3.** Explain the unscaled result. When would PCA on unscaled data be the right
choice?

*Your answer here.*

### Seeing versus measuring: t-SNE (provided)

t-SNE is a *neighbour embedding*: it places points in two dimensions so that each point's
nearest neighbours in the original space stay near it. It does not preserve distances
between groups, and its picture depends on a tuning parameter (`perplexity`) and on the
random seed. The cell runs it twice on the standardised data (about 20 seconds).

In [ ]:
Z = StandardScaler().fit_transform(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for a, perp in zip(ax, [5, 50]):
    emb = TSNE(n_components=2, perplexity=perp, init="pca", random_state=SEED).fit_transform(Z)
    for lab, col, name in [(0, "#4a90c2", "benign"), (1, "#c8102e", "malignant")]:
        a.scatter(emb[y == lab, 0], emb[y == lab, 1], s=6, alpha=0.6, c=col, label=name)
    a.set(title=f"t-SNE, perplexity {perp}", xticks=[], yticks=[])
ax[0].legend(); plt.tight_layout(); plt.show()

**Question 3.4.** Compare the two t-SNE plots with each other and with the PCA plot.
Which of the following can you read from a t-SNE plot: (a) which cases are similar to
which; (b) how far apart the two groups are; (c) how large or spread out each group is;
(d) how many groups there are? Why do the axes have no units?

*Your answer here.*

---
## Task 4 — How many components does the model need? (about 12 minutes)

Now PCA feeds a classifier, so it goes **inside the pipeline**, where cross-validation refits
the scaler and the PCA on each training fold.

**Your task.** For each `k` in `[1, 2, 3, 5, 7, 10, 15, 20, 30]`, cross-validate
`make_pipeline(StandardScaler(), PCA(n_components=k, random_state=SEED), LogisticRegression(max_iter=5000))` on
`X_train, y_train` with `cv_scores(..., metrics=("roc_auc", "accuracy"))`. Add one row for
logistic regression on all 30 standardised features (no PCA). Store the results in a
DataFrame `curve` indexed by `k`, plot mean ROC AUC against `k` with the all-features value
as a horizontal line, and print the table.

Then choose `k` with the **one-standard-error rule**: take the best mean ROC AUC, compute its
standard error (`roc_auc_sd / sqrt(5)`), and choose the *smallest* `k` whose mean ROC AUC is
within one standard error of the best. Store it as `k_chosen`.

In [ ]:
# your code here


**Question 4.1.** Where is the "knee" of the curve? How does the best PCA pipeline
compare with all 30 features, given the standard deviations across folds?

**Question 4.2.** Which `k` does the one-standard-error rule choose, and why prefer the
smallest adequate `k` over the single best mean?

**Question 4.3.** A colleague fits the scaler and PCA on all 569 cases *before* splitting,
then cross-validates only the logistic regression. Is that leakage? On WDBC the effect is
tiny. Why would it be much larger on a dataset with 20,000 features and 100 cases, or if the
reduction step used the labels?

*Your answer here.*

---
## Task 5 — One honest number for Part A (about 5 minutes)

**Your task.** Fit two pre-specified pipelines on the training rows: PCA with your
`k_chosen` components followed by logistic regression, and logistic regression on all 30
standardised features. Evaluate each **once** on `X_test`: report accuracy, ROC AUC and the
number of misclassified cases. Store the results in a DataFrame `test_a`.

In [ ]:
# your code here


**Question 5.1.** Which pipeline did better on the test set, and by how many cases?
Is that difference evidence that PCA *improves* the model? What would you report?

*Your answer here.*

# Part B — A learned representation of real proteins

---
## Task 6 — Meet the proteins (about 10 minutes)

**The biology in two paragraphs.** *Integral membrane proteins* (receptors, channels,
transporters) cross the lipid bilayer, usually as α-helices of roughly 20 mostly hydrophobic
residues; about a quarter of human proteins are of this kind. A *signal peptide* is a short
stretch (typically 15–30 residues) at the **N-terminus** of a protein, with a hydrophobic
core, that directs the protein into the secretory pathway; it is usually cut off. Secreted
proteins (hormones, antibodies, digestive enzymes) and many membrane proteins carry one.

**The data.** One row per protein. `sequence` is the amino-acid sequence (one letter per
residue); `is_transmembrane` and `has_signal_peptide` come from the UniProt keywords
Transmembrane (KW-0812) and Signal (KW-0732); `tm_evidence` and `sp_evidence` record the
strongest evidence behind the annotated feature; `uniref50` is the protein's cluster of
sequences sharing at least 50% identity. `shuffled_sequence` is the same protein with its
residues randomly permuted: identical amino-acid composition, order destroyed (used in
Task 9).

The provided cell loads the table and makes the one split used in Part B (25% test,
stratified on both labels together, `SEED = 42`).

In [ ]:
prot = pd.read_csv(DATA / "swissprot_human_2000.csv")
print(prot.shape, "| one protein per UniRef50 cluster:", prot.uniref50.is_unique)
display(prot[["accession", "protein_name", "length", "is_transmembrane", "has_signal_peptide",
              "tm_evidence", "sp_evidence"]].head())
display(pd.crosstab(prot.is_transmembrane, prot.has_signal_peptide, margins=True))
display(pd.concat({"transmembrane": prot.loc[prot.is_transmembrane == 1, "tm_evidence"].value_counts(),
                   "signal peptide": prot.loc[prot.has_signal_peptide == 1, "sp_evidence"].value_counts()},
                  axis=1).fillna(0).astype(int))

y_tm = prot.is_transmembrane.values
y_sp = prot.has_signal_peptide.values
idx_train, idx_test = train_test_split(np.arange(len(prot)), test_size=0.25,
                                       stratify=2 * y_tm + y_sp, random_state=SEED)
print("train", len(idx_train), "| test", len(idx_test))

**Your task.** Write a function `composition(seqs)` that returns an array with one row
per sequence and 20 columns: the fraction of each amino acid in the order
`AA = "ACDEFGHIKLMNPQRSTVWY"`. Build `C = composition(prot.sequence)`, check that every row
sums to 1, and print `C.shape`. This is the classic hand-engineered protein representation
(Nakashima & Nishikawa used it in 1994 to tell intracellular from extracellular proteins).

In [ ]:
# your code here


**Question 6.1.** Why does the dataset keep only one protein per UniRef50 cluster? What
would go wrong in the evaluation if close homologues (say 90% identical paralogues) could
fall on both sides of the split?

**Question 6.2.** Look at the evidence table. What fraction of the transmembrane labels rest
on "sequence analysis" rather than experiment? UniProt annotates a predicted signal peptide
when at least two of Phobius, Predotar, SignalP and TargetP agree, and annotates predicted
transmembrane regions when several tools agree and the prediction fits the protein's
function. What does that imply about what a classifier trained on these labels learns?

*Your answer here.*

---
## Task 7 — The geometry of a learned representation (about 12 minutes)

**How the embedding was made.** Each sequence was given to ESM-2 (6 layers, 8 million
parameters), a transformer trained by its authors to predict masked amino acids in tens of
millions of UniRef sequences; **no labels were involved in that pretraining, or in computing
these vectors.** For a protein of length L the model outputs one 320-dimensional vector per
residue; averaging them (mean pooling) gives one vector per protein, whatever its length.

The provided cell loads the embeddings of the real and of the shuffled sequences, aligned
row by row with `prot`, and defines a function for the **mean cosine similarity** between all
pairs of rows (1 = all vectors point the same way, 0 = unrelated directions on average).

In [ ]:
def load_embedding(name):
    e = pd.read_csv(DATA / name).set_index("accession").loc[prot.accession]
    return e.values

E = load_embedding("esm2_t6_8M_embeddings.csv.gz")
E_shuf = load_embedding("esm2_t6_8M_embeddings_shuffled.csv.gz")
print("E:", E.shape, "| E_shuf:", E_shuf.shape)


def mean_cosine(M):
    U = M / np.linalg.norm(M, axis=1, keepdims=True)
    S = U @ U.T
    n = len(M)
    return (S.sum() - n) / (n * (n - 1))

**Predict first.** Pick two unrelated proteins, say a membrane transporter and a nuclear
transcription factor. What cosine similarity do you expect between their embeddings?

**Your task.**
1. Print the mean cosine similarity of `E`, and of `E` after **centring** (subtracting the
   mean vector, `E - E.mean(axis=0)`). Do the same for the composition matrix `C`.
2. Fit `PCA()` to `E` and to standardised `C`. Print PC1's share of the variance for `E`,
   and how many components each needs to reach 90% of the variance.
3. Plot PC1 against PC2 of `E` twice, side by side, coloured by `y_tm` and by `y_sp`.

*(Prediction.)*

In [ ]:
# your code here


**Question 7.1.** What is the mean cosine similarity of the raw embeddings, and what
happens after centring? Explain what *anisotropy* means and why it matters if you use
cosine similarity to find "similar proteins".

**Question 7.2.** The raw composition vectors have a high mean cosine too. What does that
tell you about interpreting a raw cosine similarity, whatever the representation?

**Question 7.3.** How many dimensions does the embedding need for 90% of its variance? Do the
two labels separate in the PC1–PC2 plot? Does that tell you how well a classifier using all
320 dimensions will do?

*Your answer here.*

---
## Task 8 — One classifier, four representations (about 15 minutes)

A **linear probe** asks how much of a property can be read off a representation by a simple,
fixed classifier. Everything except the representation is held constant: the same pipeline
(`StandardScaler` → `LogisticRegression(max_iter=5000)`), the same training rows, the same
5-fold CV, the same metrics. Any difference is due to the representation.

The four representations:
- **length only** (`np.log(prot.length)`, one column): a baseline that knows nothing about
  the sequence except its size;
- **composition** (`C`, 20 columns): what a biologist would compute first;
- **ESM-2** (`E`, 320 columns): the learned representation;
- **ESM-2 on shuffled sequences** (`E_shuf`, 320 columns): the same model given the same
  residues in random order (Task 9 explains why).

**Predict first.** For each task (transmembrane, signal peptide), rank the four
representations by the ROC AUC you expect.

*(Prediction.)*

**Your task.** For each task and each representation, run `cv_scores` on the training
rows only (`M[idx_train]`, `y[idx_train]`). Collect mean ROC AUC, its standard deviation and
mean average precision into a DataFrame `probe` with one row per representation and
columns grouped by task. Print the training prevalence of each label (the average
precision of a random ranking).

In [ ]:
# your code here


**Question 8.1.** Describe the table. Which representation wins each task, and by how
much relative to the fold-to-fold standard deviation? How does each compare with the
length-only baseline?

**Question 8.2.** Composition does much better on one task than on the other. Using the
biology at the start of Task 6, explain why.

*Your answer here.*

---
## Task 9 — What is the embedding seeing? (about 10 minutes)

Look again at the fourth row of your table. The shuffled sequences have **exactly** the
composition of the real ones, so ESM-2 on shuffled sequences can only use composition (and
length). This is a **control experiment**: it removes one property (residue order) and keeps
everything else.

Biochemists have described the order-dependent signals for decades. The provided cell
builds two engineered features from the Kyte–Doolittle hydropathy scale (1982):
- **max window hydropathy**: the highest average hydropathy of any 19 consecutive residues
  (a membrane-spanning helix is a long hydrophobic stretch);
- **N-terminal hydropathy**: the average hydropathy of the first 25 residues (where a signal
  peptide's hydrophobic core sits).

In [ ]:
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5, "G": -0.4,
      "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, "S": -0.8,
      "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}          # Kyte & Doolittle (1982)


def max_window_hydropathy(seq, w=19):
    h = np.array([KD[a] for a in seq])
    return np.convolve(h, np.ones(w) / w, mode="valid").max()


def nterm_hydropathy(seq, n=25):
    return np.mean([KD[a] for a in seq[:n]])


H = np.array([[max_window_hydropathy(s), nterm_hydropathy(s)] for s in prot.sequence])
print(H.shape, "| first protein:", H[0].round(2))

**Your task.** Build `CH = np.hstack([C, H])` and cross-validate the same probe on `CH`
(composition plus the two hydropathy features) for both tasks, exactly as in Task 8, and add the results to `probe`
as a fifth row named `"composition + 2 hydropathy features (22)"`. Display the updated
table.

In [ ]:
# your code here


**Question 9.1.** What happens to ESM-2's advantage when the residues are shuffled?
What does that tell you about *what* the embedding contributes for these two tasks?

**Question 9.2.** How does composition plus two hand-engineered features compare with ESM-2
on each task? What does this say about the module's question, "what is the simplest
appropriate approach, and what evidence do you have that it is appropriate"?

**Question 9.3.** Recall Question 6.2. Several of the tools UniProt relies on for predicted
annotations look for exactly these hydrophobic patterns. How should that change the way you
describe your results in 9.2?

*Your answer here.*

---
## Task 10 — One honest number for Part B (about 5 minutes)

**Your task.** For each task, fit the probe on the training rows for three pre-specified
representations (composition, ESM-2, composition + 2 hydropathy features) and evaluate
each **once** on the test rows (`idx_test`): ROC AUC and average precision. Store the
results in a DataFrame `test_b`.

In [ ]:
# your code here


**Question 10.1.** Do the test results agree with cross-validation? Summarise Part B in
three sentences that a biologist could act on, including one limitation.

**Question 10.2.** ESM-2 was pretrained on UniRef sequences that almost certainly include
these 2,000 human proteins, test set included. Is that leakage? What exactly did the model see
during pretraining, and what did it not see?

*Your answer here.*

---
## Checkpoint: end of the core lab

You should now have: the explained-variance plots and k for 90% (Task 2); the unscaled-PCA
result and your loading interpretation (Task 3); the components-versus-ROC-AUC curve and
`k_chosen` (Task 4); the Part A test result (Task 5); the anisotropy and PCA results for the
embedding (Task 7); the five-row representation table (Tasks 8–9); and the Part B test
result (Task 10), with every question answered.

The extensions below are optional.

## Extension E1 — How much of the embedding does the probe need?

Put PCA *inside* the probe: `make_pipeline(StandardScaler(), PCA(n_components=k, random_state=SEED),
LogisticRegression(max_iter=5000))` on `E` for `k` in `[2, 8, 16, 32, 64, 128]`, for both
tasks (training rows, `cv_scores`). Plot ROC AUC against `k`. How many of the 320 dimensions
does each task need? Compare with your answer to 7.3.

In [ ]:
# your code here


*Your answer here.*

## Extension E2 — Which membrane proteins does the embedding miss?

The lecture showed the probe's most confident *false positives*. Here you look at the other
kind of error. Using cross-validated predictions on the training rows
(`sklearn.model_selection.cross_val_predict` with `method="predict_proba"`), list the five
**transmembrane** proteins with the *lowest* predicted probability from the ESM-2 probe, with
their length and evidence. Look one or two up on uniprot.org (Subcellular location and
Topology). Then compute the same cross-validated predictions with the composition + 2
hydropathy probe (`CH`) for those five proteins, and their max window hydropathy (`H[:, 0]`).
What do the five have in common, and why might mean pooling miss them?

In [ ]:
# your code here


*Your answer here.*

## Extension E3 — t-SNE of the embedding, and what its picture depends on

Run t-SNE on the standardised embeddings `E` at perplexity 30 twice: once with
`init="pca", random_state=SEED` and once with `init="random", random_state=1`. Colour by
`y_sp` and compare the two pictures with each other and with the PCA plot from Task 7.
Which features of the picture survive the change? Would you report "signal-peptide proteins
form a separate cluster" on this evidence? (About a minute.)

In [ ]:
# your code here


*Your answer here.*

## Final reflection

Answer in a short paragraph each.

**R1.** You are given a new biological dataset. Describe how you would decide between (a) the
raw measured features, (b) PCA of those features, and (c) a foundation-model embedding.
What evidence would you collect before choosing?

**R2.** Week 6 asked what pretraining does not fix. After today, add one thing to that list and
one thing pretraining *did* provide here.

*Your answer here.*